In [ ]:
#%pip install --proxy=proxy.kip.uni-heidelberg.de:8080 --find-links="//twix3/bolo/programs/python/phonon_monte_carlo" phonon_monte_carlo --force-reinstall

In [ ]:
#%pip install --proxy=proxy.kip.uni-heidelberg.de:8080 toml pyarrow

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "//twix3/bolometer/programs/python")

import toml
import phonon_monte_carlo

In [ ]:
# Assume that a file old_config.toml has previously been created
# Let's say you want to make a change to it
config = toml.load("old_config.toml")
config["number_of_particles"] = 1000
config["output_folder"] = "cool_example"
config["num_workers"] = 5
with open("config.toml", "w") as f:
    toml.dump(config, f)
os.makedirs("cool_example", exist_ok=True)

# Set up the particle source positions
sources_X = np.linspace(-0.005, 0.005, 101)
sources_Y = np.zeros(101)
sources_Z = np.zeros(101)
sources = pd.DataFrame({"Source X": sources_X, "Source Y": sources_Y, "Source Z": sources_Z})
sources.to_parquet("sources.parquet")

# Run simulation for all sources
phonon_monte_carlo.run("config.toml", "sources.parquet")

# Evaluate output
df = pd.read_parquet("cool_example/output.parquet")
df.head()

In [ ]:
# Need to run with `write_absorbed_energy = true` in config.toml beforehand
# otherwise the column "Energy Absorbed" won't exist
for i, row in df.iterrows():
    dt = row["Time Bin Size"]
    t = np.arange(len(row["Energy Absorbed"])) * dt
    plt.plot(t*1e6, row["Energy Absorbed"])

plt.xlabel("Time / µs")
plt.ylabel("Absorbed Energy Fraction")